In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA L4, 23034 MiB, 0 MiB


In [2]:
!pip install transformers datasets evaluate numpy torch peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00


In [3]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=70980bfcd3f654deb3a648d11117c1ae07654e5724dc7cad897fb9fa2a3dbf0c
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [4]:
from datasets import load_from_disk
from transformers import BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import BartForConditionalGeneration
import evaluate
import numpy as np
import torch
import os
import logging
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

In [5]:
"""
CNN/DailyMail 数据集预处理脚本
用于 BART 模型适配策略对比研究：全参数微调、超轻量微调、LoRA 与从零训练

预处理步骤：
1. 加载 CNN/DailyMail 数据集
2. 最小化文本清洗（仅标准化空白字符）
3. 使用 BART 分词器进行分词（最大长度 512）
4. 划分训练集、验证集和测试集
5. 保存为 Hugging Face Dataset 格式
"""

from datasets import load_dataset, DatasetDict
from transformers import BartTokenizer, BartModel
import logging
from typing import Dict, List

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

_train_size = 20000
_val_size = 5000
_test_size = 5000

class CNNDailyMailPreprocessor:
    """CNN/DailyMail 数据集预处理类"""
    
    def __init__(self, model_name: str = "facebook/bart-base", max_length: int = 1024):
        """
        初始化预处理器
        
        Args:
            model_name: BART 模型名称
            max_length: 最大序列长度
        """
        self.model_name = model_name
        self.max_length = max_length
        
        # 加载 BART 分词器
        logger.info(f"加载分词器：{model_name}")
        self.tokenizer = BartTokenizer.from_pretrained(model_name)
        
        logger.info("预处理器初始化完成")
    
    def preprocess_text(self, text: str) -> str:
        """
        最小化文本预处理（仅标准化空白字符）
        
        Args:
            text: 原始文本
            
        Returns:
            处理后的文本
        """
        if not text:
            return ""
        
        # 仅标准化空白字符，保留原始大小写、标点和数字
        text = ' '.join(text.split()).strip()
        
        return text
    
    def tokenize_function(self, examples: Dict[str, List]) -> Dict[str, List]:
        """
        分词函数
        
        Args:
            examples: 包含文章和摘要的字典
        
        Returns:
            分词后的字典
        """
        # 最小化预处理文章
        articles = [self.preprocess_text(article) for article in examples['article']]
        
        # 最小化预处理摘要
        highlights = [self.preprocess_text(highlight) for highlight in examples['highlights']]
        
        # 对文章进行分词
        model_inputs = self.tokenizer(
            articles,
            max_length=self.max_length,
            truncation=True,
            add_special_tokens=True,
            padding="max_length"
        )
        
        # 对摘要进行分词（作为标签）
        labels = self.tokenizer(
            highlights,
            max_length=self.max_length,
            truncation=True,
            add_special_tokens=True,
            padding="max_length"
        )
        
        model_inputs["labels"] = labels["input_ids"]
        
        return model_inputs
    
    def load_and_preprocess(self) -> DatasetDict:
        """
        加载并预处理数据集
        
        Returns:
            预处理后的 DatasetDict
        """
        logger.info(f"加载数据集：cnn_dailymail (版本：3.0.0)")
        
        # 加载数据集
        dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")
        
        # 子集大小（用于快速原型开发）
        train_size = min(_train_size, len(dataset['train']))
        val_size = min(_val_size, len(dataset['validation']))
        test_size = min(_test_size, len(dataset['test']))
        
        # 创建子集
        train_subset = dataset['train'].select(range(train_size))
        val_subset = dataset['validation'].select(range(val_size))
        test_subset = dataset['test'].select(range(test_size))
        
        logger.info(f"训练集大小：{len(train_subset)}")
        logger.info(f"验证集大小：{len(val_subset)}")
        logger.info(f"测试集大小：{len(test_subset)}")
        
        # 应用预处理
        logger.info("开始预处理数据集...")
        
        train_processed = train_subset.map(
            self.tokenize_function,
            batched=True,
            batch_size=8,
            remove_columns=['article', 'highlights', 'id']
        )
        
        val_processed = val_subset.map(
            self.tokenize_function,
            batched=True,
            batch_size=8,
            remove_columns=['article', 'highlights', 'id']
        )
        
        test_processed = test_subset.map(
            self.tokenize_function,
            batched=True,
            batch_size=8,
            remove_columns=['article', 'highlights', 'id']
        )
        
        # 创建 DatasetDict
        processed_datasets = DatasetDict({
            'train': train_processed,
            'validation': val_processed,
            'test': test_processed
        })
        
        logger.info("数据集预处理完成")
        
        return processed_datasets
    
    def save_dataset(self, dataset: DatasetDict, save_path: str = "./processed_cnn_dailymail"):
        """
        保存预处理后的数据集
        
        Args:
            dataset: 预处理后的数据集
            save_path: 保存路径
        """
        logger.info(f"保存数据集到：{save_path}")
        dataset.save_to_disk(save_path)
        logger.info("数据集保存完成")
    
    def get_dataset_info(self, dataset: DatasetDict):
        """显示数据集信息"""
        logger.info("数据集信息:")
        for split in dataset.keys():
            logger.info(f"  {split}: {len(dataset[split])} 个样本")
            
            if len(dataset[split]) > 0:
                sample = dataset[split][0]
                logger.info(f"    样本键：{list(sample.keys())}")
                
                # 原始 tokens (ID 列表)
                input_tokens = sample['input_ids']
                label_tokens = sample['labels']

                # 解码第一个样本查看
                input_text = self.tokenizer.decode(input_tokens, skip_special_tokens=True)
                label_text = self.tokenizer.decode(label_tokens, skip_special_tokens=True)
                
                logger.info(f"    输入原始 Tokens : {input_tokens[:]}")
                logger.info(f"    标签原始 Tokens : {label_tokens[:]}")              

                logger.info(f"    输入文本 (前 100 字符): {input_text[:100]}...")
                logger.info(f"    标签文本 (前 100 字符): {label_text[:100]}...")

def process_data():
    """主函数"""
    logger.info("开始 CNN/DailyMail 数据集预处理")
    
    # 初始化预处理器
    preprocessor = CNNDailyMailPreprocessor(
        model_name="facebook/bart-base",
        max_length=512
    )
    
    # 加载并预处理数据集
    processed_datasets = preprocessor.load_and_preprocess()
    
    # 显示数据集信息
    preprocessor.get_dataset_info(processed_datasets)
    
    # 保存数据集
    preprocessor.save_dataset(processed_datasets, "./processed_cnn_dailymail")
    
    logger.info("预处理流程完成")

process_data()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [8]:
# ==================== Configuration ====================
MODEL_NAME = "facebook/bart-base"
MAX_TARGET_LENGTH = 256
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 3e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
seed = 42

# ==================== LoRA Configuration ====================
# LoRA 超参数配置
LORA_R = 8  # LoRA 秩，通常为 4, 8, 16
LORA_ALPHA = 16  # LoRA 缩放参数
LORA_DROPOUT = 0.1  # LoRA dropout 率
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj"]  # BART 的目标模块

# 配置 LoRA
lora_config = LoraConfig(
    r=LORA_R,  # 秩
    lora_alpha=LORA_ALPHA,  # 缩放参数
    target_modules=LORA_TARGET_MODULES,  # 要应用 LoRA 的模块
    lora_dropout=LORA_DROPOUT,  # dropout
    bias="none",  # 是否训练偏置
    task_type=TaskType.SEQ_2_SEQ_LM,  # 任务类型：序列到序列语言模型
)

# ==================== Load Model & Tokenizer ====================
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
base_model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

# 应用 LoRA适配器到模型
lora_model = get_peft_model(model, lora_config)
# 打印可训练参数信息
lora_model.print_trainable_parameters()

# Ensure pad_token_id is set correctly
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.decoder_start_token_id = tokenizer.bos_token_id

lora_model.config.pad_token_id = tokenizer.pad_token_id
lora_model.config.decoder_start_token_id = tokenizer.bos_token_id

# ==================== Load Datasets ====================
# Load from single DatasetDict directory
processed_datasets = load_from_disk("./processed_cnn_dailymail")

# Extract splits
train_tokenized_data = processed_datasets['train']
train_tokenized_data = train_tokenized_data.shuffle(seed=seed).select(range(20000))

validation_tokenized_data = processed_datasets['validation']
validation_tokenized_data = validation_tokenized_data.shuffle(seed=seed).select(range(2000))

test_tokenized_data = processed_datasets['test']
test_tokenized_data = test_tokenized_data.shuffle(seed=seed).select(range(2000))

# ==================== Training Arguments ====================
def training_args(logging_name="test",NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=4,EVAL_BATCH_SIZE=4):
    return Seq2SeqTrainingArguments(
        output_dir="bart_cnndailymail_model",
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        save_strategy="epoch",
        eval_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="rougeL",
        greater_is_better=True,
        logging_dir=f"./logs/{logging_name}",
        logging_strategy="steps",
        logging_steps=1,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        generation_num_beams=2,
        fp16=torch.cuda.is_available(),
        report_to="tensorboard",
    )

# ==================== ROUGE Metric ====================
metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Replace -100 with pad_token_id for decoding
    # labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Normalize for ROUGE (optional but recommended)
    decoded_preds = ["\n".join(pred.strip().split("\n")) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split("\n")) for label in decoded_labels]
    
    # Compute ROUGE scores
    result = metric.compute(
        predictions=decoded_preds, 
        references=decoded_labels,
        use_stemmer=True,
        rouge_types=["rouge1", "rouge2", "rougeL"]
    )
    
    # Convert to percentage
    result = {key: round(value * 100, 4) for key, value in result.items()}
    
    # Add mean length for reference
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    
    return result

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 140,305,152 || trainable%: 0.6306


In [9]:
'''epoch=2,batch size: 8, Base'''
trainer_base = Seq2SeqTrainer(
    model=base_model,
    args=training_args(logging_name="base8",NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=8,EVAL_BATCH_SIZE=8),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Starting training base model...")
trainer_base.train()
# 保存模型
trainer_base.save_model("bart_cnndailymail_model_base")
print(f"Base fine-tuning completed.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training base model...


Epoch,Training Loss,Validation Loss


OverflowError: out of range integral type conversion attempted

In [ ]:
'''epoch=2,batch size: 8, LoRA'''
#LoRA
trainer_lora = Seq2SeqTrainer(
    model=lora_model,
    args=training_args(logging_name="lora8",NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=8,EVAL_BATCH_SIZE=8),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Starting training LoRA model...")
trainer_lora.train()
# 保存 LoRA 适配器权重（轻量级）
lora_model.save_pretrained("bart_cnndailymail_model_lora8")
tokenizer.save_pretrained("bart_cnndailymail_model_lora8")

print(f"LoRA fine-tuning completed.")
print(f"LoRA adapter saved to: bart_cnndailymail_model_lora8")
print(f"Trainable parameters saved (much smaller than full model)")

In [ ]:
'''epoch=2,batch size: 16, Base'''
trainer_base = Seq2SeqTrainer(
    model=base_model,
    args=training_args(logging_name="base16",NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=16,EVAL_BATCH_SIZE=16),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Starting training base model...")
trainer_base.train()
# 保存模型
trainer_base.save_model("bart_cnndailymail_model_base16")
print(f"Base fine-tuning completed.")

In [ ]:
'''epoch=2,batch size: 16, LoRA'''
#LoRA
trainer_lora = Seq2SeqTrainer(
    model=lora_model,
    args=training_args(logging_name="lora16",NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=16,EVAL_BATCH_SIZE=16),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Starting training LoRA model...")
trainer_lora.train()
# 保存 LoRA 适配器权重（轻量级）
lora_model.save_pretrained("bart_cnndailymail_model_lora16")
tokenizer.save_pretrained("bart_cnndailymail_model_lora16")

print(f"LoRA fine-tuning completed.")
print(f"LoRA adapter saved to: bart_cnndailymail_model_lora16")
print(f"Trainable parameters saved (much smaller than full model)")

Starting training LoRA model...


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.07 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.83 GiB is free. Including non-PyTorch memory, this process has 12.73 GiB memory in use. Of the allocated memory 12.34 GiB is allocated by PyTorch, and 260.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)